# Assignment 2: Learn to reconstruct images from CAMUS dataset

Olivier Bernard and Razmig Kéchichian

---
- The goal of this project is to learn to reconstruct images from CAMUS (ultrasound images) dataset.
- We are providing you with the code to load this type of data. 
- Choose the method you wish to use based on the previous Jupyter notebooks and implement your own code

```
Run the following cell to import the necessary libraries to work with the data and PyTorch.
```

In [ ]:
# import libraries
import os
import torch
import torch.nn as nn
import monai
from monai.apps import MedNISTDataset
from monai.data import DataLoader
from monai.data import Dataset as MonaiDataset
from monai.transforms import Compose, LoadImage, ToTensor
from monai.networks.nets import AutoencoderKL
from torchvision import datasets, transforms
from torch.utils.data import Dataset as TorchDataset
import numpy as np
from torch.utils.data import random_split
import torchvision
from torchinfo import summary
from tqdm.auto import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from sklearn.manifold import TSNE
from IPython.display import Image, display
%matplotlib inline

print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Monai version: {monai.__version__}")


---
## Load and Visualize the Data

```
Run the following cell to download and create Dataset from pytorch
```

In [ ]:
# Make sure the data is downloaded and extracted where it should be
if not os.path.exists("data/camus_64"):
    import zipfile
    from io import BytesIO
    from urllib.request import urlopen

    zipurl = "https://www.creatis.insa-lyon.fr/~bernard/camus/camus_64.zip"
    with urlopen(zipurl) as zipresp:
        with zipfile.ZipFile(BytesIO(zipresp.read())) as zfile:
            for member in tqdm(
                zfile.infolist(), desc="Downloading and extracting data", position=0, leave=True
            ):
                try:
                    zfile.extract(member, "data/")
                except zipfile.error as e:
                    pass


### Create list of files to create the train, valid, test datasets

```
Run the 2 cell below to split these data into training, validation and testing sets. We will use 80% of the data for training, 10% for validation and 10% for testing. The split is done by patient ID, so that the same patient will not appear in different sets
```

In [ ]:
def subdirs(
    folder: str, join: bool = True, prefix: str = None, suffix: str = None, sort: bool = True
) -> list[str]:
    """Get a list of subdirectories in a folder.

    Args:
        folder: The path to the folder.
        join: Whether to join the folder path with subdirectory names. Defaults to True.
        prefix: Filter subdirectories by prefix. Defaults to None.
        suffix: Filter subdirectories by suffix. Defaults to None.
        sort: Whether to sort the resulting list. Defaults to True.

    Returns:
        A list of subdirectory names in the given folder.
    """
    if join:
        l = os.path.join  # noqa: E741
    else:
        l = lambda x, y: y  # noqa: E731, E741
    res = [
        l(folder, i)
        for i in os.listdir(folder)
        if os.path.isdir(os.path.join(folder, i))
        and (prefix is None or i.startswith(prefix))
        and (suffix is None or i.endswith(suffix))
    ]
    if sort:
        res.sort()
    return res

In [ ]:
# Specify the data directory
data_dir = Path("data/camus_64").resolve()

# List all the patients id
keys = subdirs(data_dir, prefix="patient", join=False)

# Split the patients into 80/10/10 train/val/test sets
train_keys, val_and_test_keys = train_test_split(keys, train_size=0.8, random_state=12345)
val_keys, test_keys = train_test_split(val_and_test_keys, test_size=0.5, random_state=12345)
train_keys = sorted(train_keys)
val_keys = sorted(val_keys)
test_keys = sorted(test_keys)

# Create train, val and test datalist
viws_instants = ["2CH_ED", "2CH_ES", "4CH_ED", "4CH_ES"]
train_datalist = [
    {
        "image": str(data_dir / key / f"{key}_{view}.nii.gz"),
        "label": str(data_dir / key / f"{key}_{view}_gt.nii.gz"),
    }
    for key in train_keys
    for view in viws_instants
]
valid_datalist = [
    {
        "image": str(data_dir / key / f"{key}_{view}.nii.gz"),
        "label": str(data_dir / key / f"{key}_{view}_gt.nii.gz"),
    }
    for key in val_keys
    for view in viws_instants
]
test_datalist = [
    {
        "image": str(data_dir / key / f"{key}_{view}.nii.gz"),
        "label": str(data_dir / key / f"{key}_{view}_gt.nii.gz"),
    }
    for key in test_keys
    for view in viws_instants
]

### Create Dataset objects

```
Run the 3 cell below to create a `Dataset` object for each set. This object will be used to load the data during training and testing.
```

In [ ]:
def load_data(image_path, type_image=True):
    
    # Use LoadImage to load a nii.gz file into a tensor
    loader = LoadImage(image_only=True)  # Load image only
    image = loader(image_path)
    image = image.numpy().transpose(1, 0)
    
    # if input image in gray value, put every thing between 0 and 1
    if type_image==True:
        image = image / image.max()
        
    # if input label, make it as a one hot encoding matrix
    else:
        image = image.astype(int)
        image = np.eye(4)[image]
        # Transpose to obtain the desire shape of 4x64x64
        image = image.transpose(2, 0, 1)
    
    return image

In [ ]:
class CustomDataset(TorchDataset):
    def __init__(self, datalist, transform=None):
        self.datalist = datalist
        self.transform = transform

    def __len__(self):
        return len(self.datalist)

    def __getitem__(self, idx):
        # Load image and label from datalist
        image_path = self.datalist[idx]['image']
        label_path = self.datalist[idx]['label']

        # Load files as tensors, as required
        image = load_data(image_path, type_image=True)
        label = load_data(label_path, type_image=False)

        # Apply transformations if necessary
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# How many samples per batch to load
batch_size = 64

# convert data to torch.FloatTensor
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Choose the training and test datasets
train_data = CustomDataset(train_datalist, transform)
valid_data = CustomDataset(valid_datalist, transform)
test_data = CustomDataset(test_datalist, transform)

# Convertir les données en Dataset MONAI
train_dataset = MonaiDataset(data=[{"image": img, "label": label} for img, label in train_data], transform=Compose([ToTensor()]))
valid_dataset = MonaiDataset(data=[{"image": img, "label": label} for img, label in valid_data], transform=Compose([ToTensor()]))
test_dataset = MonaiDataset(data=[{"image": img, "label": label} for img, label in test_data], transform=Compose([ToTensor()]))

# prepare data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

print(f"Training dataset size: {len(train_loader.dataset)}")
print(f"Validation dataset size: {len(valid_loader.dataset)}")
print(f"Test dataset size: {len(test_loader.dataset)}")

### Visualize a Batch of Training Data

It's always important to check the accuracy of the data before going any further.

```
Run the cell below to display a subset of the training dataset and the size of a batch
```

In [ ]:
# obtain one batch of training images
dataiter_train = iter(train_loader)
batch = next(dataiter_train)
images = batch["image"]
images = images.numpy()
print(f"The image batch size is {images.shape}")

# plot the images in the batch, along with the corresponding labels
fig = plt.figure(figsize=(25, 4))
for idx in np.arange(20):
    ax = fig.add_subplot(2, 10, idx+1, xticks=[], yticks=[])
    ax.imshow(np.squeeze(images[idx]), cmap='gray')

### View an Image in More Detail

```
Run the cell below to display an image with the value of each pixel. This will enable you to check the operations performed by the transforms.Compose() function.
```

In [ ]:
img = np.squeeze(images[0])
print(f"The size of an image from the dataset is {img.shape}")

fig = plt.figure(figsize = (15,15)) 
ax = fig.add_subplot(111)
ax.imshow(img, cmap='gray')
width, height = img.shape
thresh = img.max()/2.5
for x in range(width):
    for y in range(height):
        val = round(img[x][y],2) if img[x][y] !=0 else 0
        ax.annotate(f"{val:.1f}", xy=(y,x),
                    horizontalalignment='center',
                    verticalalignment='center',
                    color='white' if img[x][y]<thresh else 'black')

---
## Define the Network Architecture

Your text and code

---
## Train the Network

Your text and code

---
## Evaluate the performance of our trained model on a test dataset

Your text and code